In [16]:
import importlib
import random
from pathlib import Path
from collections import Counter
import lm.preprocessing as preprocessing
from lm.preprocessing import tokenize
from config import (
    SEED,
    TRAIN_TOKEN_LIMIT,
    VALID_TOKEN_LIMIT,
    TEST_TOKEN_LIMIT,
)

DATA_DIR = Path("/home/gokay/Desktop/dataset")


TRAIN_DIR = DATA_DIR / "train" / "train"
VALID_DIR = DATA_DIR / "validation" / "validation"
TEST_DIR = DATA_DIR / "test" / "test"

In [2]:
# --- Train Data ---
train_files = sorted(TRAIN_DIR.glob("*.txt"))
print("Train data size:", len(train_files))
with train_files[0].open(encoding="utf-8") as file:
    print(file.read(1000))

Train data size: 28602














The Old Testament of the King James Version of the Bible




The First Book of Moses:  Called Genesis


1:1 In the beginning God created the heaven and the earth.

1:2 And the earth was without form, and void; and darkness was upon
the face of the deep. And the Spirit of God moved upon the face of the
waters.

1:3 And God said, Let there be light: and there was light.

1:4 And God saw the light, that it was good: and God divided the light
from the darkness.

1:5 And God called the light Day, and the darkness he called Night.
And the evening and the morning were the first day.

1:6 And God said, Let there be a firmament in the midst of the waters,
and let it divide the waters from the waters.

1:7 And God made the firmament, and divided the waters which were
under the firmament from the waters which were above the firmament:
and it was so.

1:8 And God called the firmament Heaven. And the evening and the
morning were the second day.

1:9 And God said

In [3]:
# --- Test Data ---
test_files = sorted(TEST_DIR.glob("*.txt"))
print("Test data size:", len(test_files))
with test_files[0].open(encoding="utf-8") as file:
    print(file.read(1000))

Test data size: 100
ST. PAUL***


E-text prepared by Josephine Paolucci and the Project Gutenberg Online
Distributed Proofreading Team



REMINISCENCES OF PIONEER DAYS IN ST. PAUL

A Collection of Articles Written for and Published in the Daily
Pioneer Press.

By FRANK MOORE






NEWSPAPER STRUGGLES OF PIONEER DAYS.

A BRIEF NARRATION OF INCIDENTS AND EVENTS CONNECTED WITH THE EARLY
DAYS OF ST. PAUL, DAILY NEWSPAPERS.


If James M. Goodhue could revisit the earth and make a tour among the
daily newspaper offices of St. Paul he would discover that wonderful
strides had been made in the method of producing a newspaper during
the latter half of the past century. Among the first things to attract
the attention of this old-timer would be the web-perfecting press,
capable of producing 25,000 impressions an hour, instead of the old
hand press of 240 impressions an hour; the linotype machine, capable
of setting 6,000 to 10,000 ems per hour, instead of the old hand
compositor producing only 80

In [4]:
# --- Validation Data ---
valid_files = sorted(VALID_DIR.glob("*.txt"))
print("Validation data size:", len(valid_files))
with valid_files[0].open(encoding="utf-8") as file:
    print(file.read(1000))

Validation data size: 50
ST. PAUL***


E-text prepared by Josephine Paolucci and the Project Gutenberg Online
Distributed Proofreading Team



REMINISCENCES OF PIONEER DAYS IN ST. PAUL

A Collection of Articles Written for and Published in the Daily
Pioneer Press.

By FRANK MOORE






NEWSPAPER STRUGGLES OF PIONEER DAYS.

A BRIEF NARRATION OF INCIDENTS AND EVENTS CONNECTED WITH THE EARLY
DAYS OF ST. PAUL, DAILY NEWSPAPERS.


If James M. Goodhue could revisit the earth and make a tour among the
daily newspaper offices of St. Paul he would discover that wonderful
strides had been made in the method of producing a newspaper during
the latter half of the past century. Among the first things to attract
the attention of this old-timer would be the web-perfecting press,
capable of producing 25,000 impressions an hour, instead of the old
hand press of 240 impressions an hour; the linotype machine, capable
of setting 6,000 to 10,000 ems per hour, instead of the old hand
compositor producing on

In [9]:
def select_subset(file_paths, token_limit, seed):
    # Sıralama + sabit tohum, aynı kitapların seçilmesini sağlar.
    shuffled_files = sorted(file_paths)
    random.Random(seed).shuffle(shuffled_files)
    documents = []
    total_tokens = 0
    for file_path in shuffled_files:
        if total_tokens >= token_limit:
            break
        with file_path.open(encoding="utf-8") as file:
            tokens = tokenize(file.read())
        remaining = token_limit - total_tokens
        tokens = tokens[:remaining]
        if not tokens:
            continue
        documents.append({
            "book_id": file_path.stem,
            "tokens": tokens,
        })
        total_tokens += len(tokens)
    return documents

In [10]:
train_documents = select_subset(
    train_files, TRAIN_TOKEN_LIMIT, SEED
)
valid_documents = select_subset(
    valid_files, VALID_TOKEN_LIMIT, SEED
)
test_documents = select_subset(
    test_files, TEST_TOKEN_LIMIT, SEED
)
for name, documents in [
    ("Train", train_documents),
    ("Validation", valid_documents),
    ("Test", test_documents),
]:
    token_count = sum(len(doc["tokens"]) for doc in documents)
    print(
        f"{name}: {len(documents)} kitap, "
        f"{token_count:,} token"
    )

Train: 19 kitap, 1,000,000 token
Validation: 1 kitap, 100,000 token
Test: 3 kitap, 100,000 token


In [11]:
word_counts = Counter()

for document in train_documents:
    word_counts.update(document["tokens"])

print("Toplam eğitim token'ı:", sum(word_counts.values()))
print("Farklı token:", len(word_counts))
print("En sık 20 token:")
print(word_counts.most_common(20))

Toplam eğitim token'ı: 1000000
Farklı token: 39168
En sık 20 token:
[(',', 62441), ('the', 50405), ('-', 45225), ('.', 44768), ('of', 31074), ('and', 24154), ('to', 19266), ('a', 16140), ('in', 14764), ('"', 13355), ('|', 13108), ('that', 8576), ('was', 7569), ('it', 7556), ('his', 7413), ('he', 7399), ('is', 6732), ('with', 5813), ('as', 5725), ('for', 5440)]


In [12]:
MIN_FREQ = 2
MAX_VOCAB_SIZE = 20_000

special_tokens = {"<unk>", "<bos>", "<eos>"}
normal_token_limit = MAX_VOCAB_SIZE - len(special_tokens)

vocab = {
    token
    for token, count in word_counts.most_common(normal_token_limit)
    if count >= MIN_FREQ
}

vocab.update(special_tokens)

print("Sözlük boyutu:", len(vocab))

Sözlük boyutu: 20000


In [13]:
total_tokens = sum(word_counts.values())

unknown_count = sum(
    count
    for token, count in word_counts.items()
    if token not in vocab
)

print("Train'de <unk> olacak token sayısı:", unknown_count)
print(f"<unk> oranı: %{100 * unknown_count / total_tokens:.2f}")

Train'de <unk> olacak token sayısı: 21879
<unk> oranı: %2.19


In [19]:
importlib.reload(preprocessing)

example_vocab = {"the", "cat", "<unk>"}
example_tokens = ["the", "purple", "cat"]

print(preprocessing.replace_unknown(example_tokens, example_vocab))

['the', '<unk>', 'cat']


In [20]:
def prepare_documents(documents, vocab):
    return [
        {
            "book_id": document["book_id"],
            "tokens": preprocessing.replace_unknown(
                document["tokens"], vocab
            ),
        }
        for document in documents
    ]


train_ready = prepare_documents(train_documents, vocab)
valid_ready = prepare_documents(valid_documents, vocab)
test_ready = prepare_documents(test_documents, vocab)

In [22]:
import json
from pathlib import Path

OUTPUT_DIR = Path("data") / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    "train": train_ready,
    "validation": valid_ready,
    "test": test_ready,
}

for name, documents in datasets.items():
    output_path = OUTPUT_DIR / f"{name}.json"

    with output_path.open("w", encoding="utf-8") as file:
        json.dump(documents, file, ensure_ascii=False)

with (OUTPUT_DIR / "vocab.json").open("w", encoding="utf-8") as file:
    json.dump(sorted(vocab), file, ensure_ascii=False)

print("Veriler ve sözlük kaydedildi:", OUTPUT_DIR.resolve())

Veriler ve sözlük kaydedildi: /home/gokay/PycharmProjects/nlp_homework/data/processed
